# PubHealth Dataset — First Analysis
ASHO-UAB Fake News Prediction Project

In [2]:
# Install dependencies
!pip install datasets pandas matplotlib seaborn -q


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

# Load dataset
ds = load_dataset('bigbio/pubhealth', trust_remote_code=True)
print(ds)

HTTPError: HTTP Error 404: Not Found

In [ ]:
# Convert splits to DataFrames
train_df = ds['train'].to_pandas()
val_df   = ds['validation'].to_pandas()
test_df  = ds['test'].to_pandas()

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
train_df.head(3)

In [ ]:
# Basic info
print(train_df.dtypes)
print("\nMissing values (train):")
print(train_df.isnull().sum())

## 1. Label Distribution

In [ ]:
label_map = {0: 'true', 1: 'false', 2: 'unproven', 3: 'mixture'}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, df) in zip(axes, [('Train', train_df), ('Val', val_df), ('Test', test_df)]):
    counts = df['label'].map(label_map).value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=ax, palette='Set2')
    ax.set_title(f'{name} split')
    ax.set_ylabel('Count')
    for p in ax.patches:
        ax.annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()), ha='center', va='bottom')
plt.suptitle('Label Distribution per Split', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Text Length Analysis

In [ ]:
for col in ['claim', 'main_text', 'explanation']:
    train_df[f'{col}_len'] = train_df[col].fillna('').apply(lambda x: len(x.split()))

train_df[['claim_len', 'main_text_len', 'explanation_len']].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['claim_len', 'main_text_len', 'explanation_len']):
    train_df[col].clip(upper=train_df[col].quantile(0.98)).hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col.replace('_len', '').replace('_', ' ').title() + ' Length (words)')
    ax.set_xlabel('Words')
plt.suptitle('Token Length Distributions (clipped at 98th percentile)', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Length vs Label

In [ ]:
train_df['label_name'] = train_df['label'].map(label_map)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['claim_len', 'main_text_len']):
    sns.boxplot(data=train_df, x='label_name', y=col, ax=ax, palette='Set2',
                order=['true','false','mixture','unproven'],
                showfliers=False)
    ax.set_title(f'{col} by label')
plt.tight_layout()
plt.show()

## 4. Missing Data Heatmap

In [ ]:
cols_of_interest = ['claim', 'main_text', 'explanation', 'label']
missing = pd.DataFrame({
    split: df[cols_of_interest].isnull().mean() * 100
    for split, df in [('train', train_df), ('val', val_df), ('test', test_df)]
})

plt.figure(figsize=(6, 3))
sns.heatmap(missing, annot=True, fmt='.1f', cmap='Reds', cbar_kws={'label': '% missing'})
plt.title('Missing Values (%) per Split')
plt.tight_layout()
plt.show()

## 5. Sample Claims per Label

In [ ]:
for label in ['true', 'false', 'mixture', 'unproven']:
    sample = train_df[train_df['label_name'] == label]['claim'].dropna().sample(1, random_state=42).values[0]
    print(f"[{label.upper()}] {sample[:200]}\n")

## Key Takeaways
- **Class imbalance**: `false` dominates; `mixture` and `unproven` are underrepresented → use Macro F1.
- **Long main_text**: median ~300 words, some > 1000 → truncation strategy needed for transformers (512 token limit).
- **Missing data**: `main_text` and `explanation` may have nulls → decide on fallback (empty string / drop).
- **Claim length**: short (~20 words on average) → concatenating claim + main_text is feasible.